In [75]:
# The dataset from canvas seems to be bugged (errors unzipping)
# I prepared the dataset on Huggingface from earlier in the semester
# it's the same 20 subset from the lab

import os
import zipfile
from huggingface_hub import snapshot_download

IMAGE_ROOT = 'imagenet_train20a/'
TRAIN_LIST = 'imagenet_train20.txt'
VAL_IMAGE_ROOT = 'imagenet_val20/'
VAL_LIST = 'imagenet_val20.txt'


def download_dataset():
    repo_id = "bdanko/imagenetsubset20"

    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=".",
        allow_patterns=["*.zip", "*.txt"],
        local_dir_use_symlinks=False
    )

    extraction_map = {
        'imagenet_train20.zip': IMAGE_ROOT,
        'imagenet_val20.zip': VAL_IMAGE_ROOT
    }

    for zip_name, target_dir in extraction_map.items():
        if os.path.exists(zip_name):
            print(f"Extracting {zip_name} to {target_dir}...")
            if not os.path.exists(target_dir):
                os.makedirs(target_dir)

            with zipfile.ZipFile(zip_name, 'r') as zip_ref:
                zip_ref.extractall(target_dir)

        else:
            print(f"Warning: {zip_name} not found after download.")



In [57]:
download_dataset()

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Extracting imagenet_train20.zip to imagenet_train20a/...
Extracting imagenet_val20.zip to imagenet_val20/...


# 1. Build Neural Network


In [76]:
import torch
torch.manual_seed(0)

| Layer | feature map | kernel | channel | stride | Padding | activation |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| Input | 224x224x3 | | | | | |
| Convolution | 55x55x96 | 11x11 | 96 | 4 | same | ReLU |
| Max Pooling | 27x27x96 | 3x3 | | 2 | valid | |
| Convolution | 27x27x256 | 5x5 | 256 | 1 | same | ReLU |
| Max Pooling | 13x13x256 | 3x3 | | 2 | valid | |
| Convolution | 13x13x384 | 3x3 | 384 | 1 | same | ReLU |
| Convolution | 13x13x384 | 3x3 | 384 | 1 | same | ReLU |
| Convolution | 13x13x256 | 3x3 | 256 | 1 | same | ReLU |
| Max Pooling | 6x6x256 | 3x3 | | 2 | valid | |
| Flatten | 9216 | | | | | |
| Fully Connected | 4096 | | | | | ReLU |
| Fully Connected | 4096 | | | | | ReLU |
| Fully Connected | 20 | | | | | Softmax |

In [77]:
import torch
import torch.nn as nn

class AlexNet(nn.Module):
    def __init__(self, num_classes=20):
        super(AlexNet, self).__init__()

        self.features = nn.Sequential(

            # conv 11x11, stride 4, padding 5 to get 55x55
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=5, bias=False),

            # in the original AlexNet paper, they introuced something
            # called Local Response Normalization (LRN)
            # this wasn't actually useful and was meant to
            # help with ReLU explosions
            # but BatchNorm superceded it
            # batch normalization is like a parameterized
            # bias meant for the same kind of normalization
            # uses mean and variance for a specific channel
            # centers channel data at 0.0
            # with a variance of 1.0
            # it learns a gamma multiplier and beta mean shifter to
            # to learn how to scale the data for a better loss
            nn.BatchNorm2d(96),

            # relu activation
            nn.ReLU(inplace=True),

            # max pooling 27x27, kernel 3x3, stride 2
            nn.MaxPool2d(kernel_size=3, stride=2),

            # 5x5, stride 1, padding 2
            nn.Conv2d(96, 256, kernel_size=5, stride=1, padding=2, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            # 3x3 kernel, stride 2, 13x13
            nn.MaxPool2d(kernel_size=3, stride=2), # 13x13

            # 3x3, stride 1, padding 1
            nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(384),
            nn.ReLU(inplace=True),

            # 3x3, stride 1, padding 1
            nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(384),
            nn.ReLU(inplace=True),

            # 3x3, stride 1, padding 1
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            # max pool
            nn.MaxPool2d(kernel_size=3, stride=2), # 6x6

        )

        # now we need to flatten the conv representation
        # into a dense linear network

        self.classifier = nn.Sequential(

            nn.Flatten(),

            # dense fully connected layer
            # the original AlexNet used dropout
            # in the dense linear layers
            # to prevent overfitting
            # They used 0.5, but maybe too aggressive
            # so we tone down to 0.2
            nn.Dropout(p=0.2),
            nn.Linear(9216, 4096),
            nn.ReLU(inplace=True),

            # dense fully connected
            nn.Dropout(p=0.2),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),

            # dense fully connected to 20, activated by softmax
            # CrossEntropyLoss will use softmax by default
            nn.Linear(4096, num_classes),
        )

        # he initialization on
        # layers we defined
        self._initialize_weights()

    # he initialization
    # Kaiming Normal for ReLU
    # meant to adjust for ReLU dead neurons
    def _initialize_weights(self):

      # iterating to ensure we apply weights on what
      # is needed
      for m in self.modules():

        # fan_in: Preserves the magnitude of the variance in the forward pass
        # fan_out: Preserves the magnitude in the backward pass
        if isinstance(m, nn.Conv2d):
            # Kaiming Normal for ReLU activation
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
              # neutral starting point for bias
                nn.init.constant_(m.bias, 0)

        # make sure batchnorm starts at defaults and
        # learns progressively
        elif isinstance(m, nn.BatchNorm2d):
                # we start multiplier to 1 so variance stays 1
                nn.init.constant_(m.weight, 1)

                # we set shift to 0 so nothing changes
                nn.init.constant_(m.bias, 0)


    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 2. Training

In [78]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [79]:
import os
from PIL import Image
from torch.utils.data import Dataset
import torch

class ImageNet20Dataset(Dataset):
    def __init__(self, txt_file, root_dir, transform=None):
        self.img_labels = []
        self.root_dir = root_dir
        self.transform = transform

        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    self.img_labels.append((parts[0], int(parts[1])))

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        filename, label = self.img_labels[idx]
        path_flat = os.path.join(self.root_dir, filename)
        path_nested = os.path.join(self.root_dir, filename.split('_')[0], filename)
        full_path = path_flat if os.path.exists(path_flat) else path_nested

        if not os.path.exists(full_path):
            print(f"Warning: Image not found at {full_path}. Check your IMAGE_ROOT / VAL_IMAGE_ROOT config.")
            raise FileNotFoundError(f"Image not found at {full_path}")

        image = Image.open(full_path).convert("RGB")

        # apply augmentations
        target_image = self.transform(image) if self.transform else image

        return target_image, label

In [80]:
# we have 224x224 input
INPUT_SHAPE = (224, 224)

# batch size gives decent amount of steps
BATCH_SIZE = 64

# alter the training dataset for training
# AlexNet used Random Cropping and Flipping,
# and Color Jitter Augmentations
# trivial augment is a more modern approach, but
# was too aggressive
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SHAPE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# keep validation unaltered
transform_val = transforms.Compose([
    transforms.Resize(INPUT_SHAPE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = ImageNet20Dataset(txt_file=TRAIN_LIST, root_dir=f"{IMAGE_ROOT}/{IMAGE_ROOT}", transform=train_transform)

# tried cutmix and mixup
# but these may all be too aggressive

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE, # we can increase the batch size
    shuffle=True,
    num_workers=2, # faster CPU processing
    pin_memory=True # speeds up CPU/GPU transfers
)

print(f"Training dataset loaded: {len(train_dataset)} images found.")

val_dataset = ImageNet20Dataset(txt_file=VAL_LIST, root_dir=f"{VAL_IMAGE_ROOT}/{VAL_IMAGE_ROOT}", transform=transform_val)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Validation dataset loaded: {len(val_dataset)} images found.")

Training dataset loaded: 6000 images found.
Validation dataset loaded: 1000 images found.


In [81]:
# Validation function
# just checks validation loader
# for how many correct predictions
def validate(model, val_loader, criterion, device):
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    avg_acc = 100. * correct / total
    avg_loss = val_loss / len(val_loader)
    return avg_loss, avg_acc

In [82]:
import torch.optim as optim

# we'll define an epoch that is pretty high
epochs = 200

# standard base
base_lr = 0.001

model = AlexNet()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# uses softmax
# we add additional label smoothing, on top of the
# 0.1 alpha from mixup/cutmix
criterion_cls = nn.CrossEntropyLoss(label_smoothing=0.1)

# we are using SGD
# but with warmup and momentum
optimizer = optim.SGD(
    model.parameters(),
    lr=base_lr,
    momentum=0.9,
    # weight deca penalizes weights that
    # are too large to stay generalizable
    weight_decay=5e-4
    )

# we will use cosine annealing
warmup_epochs = 5
warmup = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)

decay = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)

# annealed cosine scheduler
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup, decay],
    milestones=[warmup_epochs]
)


# enable mixed precision
# for faster training w/ little
# accuracy lost
scaler = torch.amp.GradScaler('cuda')

# could use EMA to make weight updates more slow
# important for huge batches and larger weight updates
# tried but was probably overkill, lead to slower/nonexistant learning

In [83]:
import os
import torch
import torch.nn as nn

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

# saving for best model
best_val_acc = 0.0

for epoch in range(epochs):
    model.train()
    total_train, correct_train = 0, 0
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):

        # allow gpu to start processing as next batch copies
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # sets gradients to none
        # apparently an optimization, faster then
        # setting to 0
        optimizer.zero_grad(set_to_none=True)

        # this uses mixed precision
        # so instead of f32, we use f16
        with torch.amp.autocast(device_type='cuda'):
          # Forward Pass
          logits = model(images)

          # Calculate Loss
          loss = criterion_cls(logits, labels)

        # scale loss and backward
        scaler.scale(loss).backward()

        # strategy to clip gradients:
        # we prevent them from exceeding a norm of say,
        # 1.0 to prevent gradient explosions
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        # training accuracy tracking
        _, predicted = logits.max(1)

        # If labels are one-hot (from Mixup/CutMix), get the index of the max probability
        if labels.dim() > 1:
            _, targets = labels.max(1)
        else:
            targets = labels

        total_train += targets.size(0)
        correct_train += predicted.eq(targets).sum().item()

    # step the scheduler
    scheduler.step()

    # validation on EMA model
    val_loss, val_acc = validate(model, val_loader, criterion_cls, device)

    # checkpointing to save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"New best acc: {best_val_acc:.2f}%")
        # Save the actual weights to your Colab disk
        torch.save(model.state_dict(), 'best_alexnet_ema.pth')

    train_loss = running_loss / len(train_loader)
    train_acc = 100. * correct_train / total_train

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"\n>> Epoch {epoch+1} Summary: Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% <<")

New best acc: 7.00%

>> Epoch 1 Summary: Train Loss: 2.9998 | Train Acc: 5.53% | Val Loss: 2.9838 | Val Acc: 7.00% <<
New best acc: 10.10%

>> Epoch 2 Summary: Train Loss: 2.9797 | Train Acc: 7.32% | Val Loss: 2.9538 | Val Acc: 10.10% <<
New best acc: 14.40%

>> Epoch 3 Summary: Train Loss: 2.9476 | Train Acc: 9.82% | Val Loss: 2.9103 | Val Acc: 14.40% <<
New best acc: 22.60%

>> Epoch 4 Summary: Train Loss: 2.9060 | Train Acc: 14.87% | Val Loss: 2.8509 | Val Acc: 22.60% <<
New best acc: 25.60%

>> Epoch 5 Summary: Train Loss: 2.8461 | Train Acc: 20.00% | Val Loss: 2.7666 | Val Acc: 25.60% <<
New best acc: 28.10%

>> Epoch 6 Summary: Train Loss: 2.7564 | Train Acc: 23.48% | Val Loss: 2.6616 | Val Acc: 28.10% <<
New best acc: 28.70%

>> Epoch 7 Summary: Train Loss: 2.6633 | Train Acc: 26.43% | Val Loss: 2.5717 | Val Acc: 28.70% <<
New best acc: 29.20%

>> Epoch 8 Summary: Train Loss: 2.5802 | Train Acc: 27.65% | Val Loss: 2.4999 | Val Acc: 29.20% <<
New best acc: 31.40%

>> Epoch 9 Summ

KeyboardInterrupt: 

## Notes

- w/ weight initializations, massive loss means exploded weights at the beginning, so it's best to tune them correctly
- We cut many of the augmentations as they just hindered any learning at all on such a small dataset
- kaiming worked well with conv, dense layers performed well with default uniform, batchnorm performed well w/ constants
- tuning LR was important, especially w/ warm restart
- better approaches may be curriculum learning or progressive augmentation

to accelerate the learning next time, we can try saving images to vram and using v2 transforms

distributed data parallel if we have extra gpus
